# 01. Extração, Filtragem e Cruzamento de Dados

Este notebook consolida toda a etapa de preparação dos dados para o projeto de previsão do IDEB. O fluxo percorre as seguintes etapas:

1. **Carregamento dos microdados do Censo Escolar** (2015, 2017, 2019, 2021 e 2023) e filtragem por estado (Paraíba).
2. **Carregamento dos dados do IDEB** (anos iniciais, por escola) e filtragem por estado.
3. **Investigação exploratória** dos tipos de dados e formatos das colunas-chave.
4. **Limpeza e padronização** das colunas numéricas do IDEB.
5. **Cruzamento (merge)** entre Censo e IDEB, ano a ano, formando um painel longitudinal.
6. **Criação da variável alvo** `ATINGIU_META` e exportação do dataset final.

As fontes de dados são públicas, disponibilizadas pelo [INEP](https://www.gov.br/inep):
- **Censo Escolar**: microdados com informações de infraestrutura, docentes, matrículas e contexto de cada escola.
- **IDEB**: nota observada e meta projetada por escola, calculadas bienalmente.

In [1]:
import os
import sys
import pandas as pd
import numpy as np

# Adiciona o diretório raiz do projeto ao path para importar o pacote src
sys.path.insert(0, os.path.abspath('..'))

from src.carregamento_dados import encontrar_coluna_uf, carregar_censo, carregar_ideb, filtrar_por_uf
from src.preprocessamento import preparar_ideb, fazer_merge_ano, montar_painel

# Caminhos
DADOS_BRUTOS = os.path.abspath('../dados_brutos')
DADOS_TRATADOS = os.path.abspath('../dados_tratados')

# Anos do Censo Escolar disponíveis
ANOS = [2015, 2017, 2019, 2021, 2023]

print('Imports e configurações carregados com sucesso.')

Imports e configurações carregados com sucesso.


---
## 1. Carregamento e Filtragem do Censo Escolar

Os microdados do Censo Escolar são distribuídos pelo INEP em formato CSV, com separador `;` e encoding `latin1` (ISO-8859-1). Cada arquivo contém centenas de milhares de registros referentes a todas as escolas do Brasil.

Para reduzir o volume de dados e focar na análise regional, filtramos apenas as escolas do estado da **Paraíba (PB)**, identificando automaticamente a coluna de UF em cada arquivo (tipicamente `SG_UF`).

In [2]:
# Mapeia os arquivos brutos do Censo por ano
CENSOS_BRUTOS = {
    2015: 'microdados_ed_basica_2015.csv',
    2017: 'microdados_ed_basica_2017.csv',
    2019: 'microdados_ed_basica_2019.csv',
    2021: 'microdados_ed_basica_2021.csv',
    2023: 'microdados_ed_basica_2023.csv',
}

censos_pb = {}  # Dicionário {ano: DataFrame filtrado}

for ano, arquivo in sorted(CENSOS_BRUTOS.items()):
    caminho = os.path.join(DADOS_BRUTOS, arquivo)
    print(f'Carregando Censo {ano}...', end=' ')
    df = carregar_censo(caminho)
    
    # Identifica a coluna de UF
    coluna_uf = encontrar_coluna_uf(df.columns)
    print(f'Coluna UF: {coluna_uf} | Shape bruto: {df.shape}', end=' -> ')
    
    # Filtra por Paraíba
    df_pb = filtrar_por_uf(df, coluna_uf, 'PB')
    print(f'Shape PB: {df_pb.shape}')
    
    # Salva o CSV filtrado
    caminho_saida = os.path.join(DADOS_TRATADOS, f'censo_pb_{ano}.csv')
    df_pb.to_csv(caminho_saida, index=False, encoding='utf-8')
    
    censos_pb[ano] = df_pb

print(f'\nTotal de anos carregados: {len(censos_pb)}')

Carregando Censo 2015... Coluna UF: SG_UF | Shape bruto: (237879, 370) -> Shape PB: (7135, 370)
Carregando Censo 2017... Coluna UF: SG_UF | Shape bruto: (236481, 370) -> Shape PB: (6901, 370)
Carregando Censo 2019... Coluna UF: SG_UF | Shape bruto: (228521, 370) -> Shape PB: (6508, 370)
Carregando Censo 2021... Coluna UF: SG_UF | Shape bruto: (221140, 370) -> Shape PB: (5638, 370)
Carregando Censo 2023... Coluna UF: SG_UF | Shape bruto: (217625, 408) -> Shape PB: (5479, 408)

Total de anos carregados: 5


---
## 2. Carregamento e Filtragem do IDEB

O arquivo do IDEB é um Excel (`.xlsx`) que contém as notas observadas e metas projetadas de **todas** as escolas do Brasil, com colunas separadas por edição bienal (2005, 2007, ..., 2023). O cabeçalho real dos dados começa apenas na **linha 10** do arquivo (as primeiras 9 linhas contêm informações textuais do INEP).

A coluna identificadora da escola se chama `ID_ESCOLA` (diferente do Censo, que usa `CO_ENTIDADE` — mesma informação, nomes distintos).

In [3]:
# Carregar IDEB
caminho_ideb = os.path.join(DADOS_BRUTOS, 'divulgacao_anos_iniciais_escolas_2023.xlsx')
print('Carregando IDEB...', end=' ')
df_ideb_bruto = carregar_ideb(caminho_ideb)
print(f'Shape bruto: {df_ideb_bruto.shape}')

# Filtrar por Paraíba
coluna_uf_ideb = encontrar_coluna_uf(df_ideb_bruto.columns)
print(f'Coluna UF identificada: {coluna_uf_ideb}')

df_ideb_pb = filtrar_por_uf(df_ideb_bruto, coluna_uf_ideb, 'PB')
print(f'Shape após filtro (PB): {df_ideb_pb.shape}')

# Salvar
df_ideb_pb.to_csv(os.path.join(DADOS_TRATADOS, 'ideb_pb.csv'), index=False, encoding='utf-8')

Carregando IDEB... Shape bruto: (64921, 124)
Coluna UF identificada: SG_UF
Shape após filtro (PB): (1899, 124)


---
## 3. Investigação Exploratória dos Dados

Antes de realizar o cruzamento, é fundamental verificar:
- O **tipo de dado** das colunas de nota (`VL_OBSERVADO_*`, `VL_PROJECAO_*`): elas podem estar como string (object) com separadores inconsistentes ou valores textuais para ausência (`-`, `*`, `ND`).
- A **compatibilidade das chaves** de junção (`CO_ENTIDADE` no Censo vs `ID_ESCOLA` no IDEB): se os tipos forem diferentes (int vs float), o merge pode falhar silenciosamente.

In [4]:
# Tipos das colunas de notas do IDEB
colunas_obs = [col for col in df_ideb_pb.columns if col.startswith('VL_OBSERVADO_')]
colunas_proj = [col for col in df_ideb_pb.columns if col.startswith('VL_PROJECAO_')]

print('Dtypes VL_OBSERVADO_*:')
print(df_ideb_pb[colunas_obs].dtypes)
print()
print('Dtypes VL_PROJECAO_*:')
print(df_ideb_pb[colunas_proj].dtypes)

Dtypes VL_OBSERVADO_*:
VL_OBSERVADO_2005    object
VL_OBSERVADO_2007    object
VL_OBSERVADO_2009    object
VL_OBSERVADO_2011    object
VL_OBSERVADO_2013    object
VL_OBSERVADO_2015    object
VL_OBSERVADO_2017    object
VL_OBSERVADO_2019    object
VL_OBSERVADO_2021    object
VL_OBSERVADO_2023    object
dtype: object

Dtypes VL_PROJECAO_*:
VL_PROJECAO_2007    object
VL_PROJECAO_2009    object
VL_PROJECAO_2011    object
VL_PROJECAO_2013    object
VL_PROJECAO_2015    object
VL_PROJECAO_2017    object
VL_PROJECAO_2019    object
VL_PROJECAO_2021    object
dtype: object


In [5]:
# Exemplos de valores reais
print('5 exemplos de VL_OBSERVADO_2023:')
print(df_ideb_pb['VL_OBSERVADO_2023'].head(5).tolist())
print()
print('5 exemplos de VL_PROJECAO_2021:')
print(df_ideb_pb['VL_PROJECAO_2021'].head(5).tolist())

5 exemplos de VL_OBSERVADO_2023:
[7, 6.5, '-', '-', '-']

5 exemplos de VL_PROJECAO_2021:
[6.1, 5.2, 6, 4.6, 4.9]


In [6]:
# Compatibilidade das chaves de merge
print(f'CO_ENTIDADE (Censo 2023) -> dtype: {censos_pb[2023]["CO_ENTIDADE"].dtype}')
print(f'Exemplos: {censos_pb[2023]["CO_ENTIDADE"].head(3).tolist()}')
print()
print(f'ID_ESCOLA (IDEB) -> dtype: {df_ideb_pb["ID_ESCOLA"].dtype}')
print(f'Exemplos: {df_ideb_pb["ID_ESCOLA"].head(3).tolist()}')

CO_ENTIDADE (Censo 2023) -> dtype: int64
Exemplos: [25000012, 25000020, 25000047]

ID_ESCOLA (IDEB) -> dtype: float64
Exemplos: [25033158.0, 25033557.0, 25033670.0]


### Conclusões da Investigação

- As colunas `VL_OBSERVADO_*` e `VL_PROJECAO_*` estão como **strings (object)**, contendo hífens (`-`), asteriscos (`*`) e `ND` como indicadores de ausência. Precisam ser convertidas para `float`.
- `CO_ENTIDADE` (Censo) é `int64`, enquanto `ID_ESCOLA` (IDEB) é `float64`. Antes do merge, precisamos **remover nulos** em `ID_ESCOLA`, renomear para `CO_ENTIDADE` e converter para `int64`.

---
## 4. Preparação do IDEB para o Merge

A função `preparar_ideb()` executa três operações:
1. Remove linhas onde `ID_ESCOLA` é `NaN` (escolas sem código identificador não podem ser cruzadas).
2. Renomeia `ID_ESCOLA` para `CO_ENTIDADE` e converte para `int64`.
3. Converte todas as colunas `VL_OBSERVADO_*` e `VL_PROJECAO_*` para `float`, tratando os caracteres de ausência.

In [7]:
df_ideb_limpo, qtd_removidas = preparar_ideb(df_ideb_pb)

print(f'Linhas removidas por ID_ESCOLA nulo: {qtd_removidas}')
print(f'Shape do IDEB limpo: {df_ideb_limpo.shape}')
print(f'Dtype CO_ENTIDADE (após conversão): {df_ideb_limpo["CO_ENTIDADE"].dtype}')

Linhas removidas por ID_ESCOLA nulo: 0
Shape do IDEB limpo: (1899, 124)
Dtype CO_ENTIDADE (após conversão): int64


---
## 5. Cruzamento Censo x IDEB (Merge por Ano)

Para cada ano do Censo (2015, 2017, 2019, 2021, 2023), a função `fazer_merge_ano()` realiza:
1. Seleção das colunas `VL_OBSERVADO_{ano}` (renomeada para `IDEB`) e `VL_PROJECAO_{ano}` (renomeada para `IDEB_META`, quando disponível) do IDEB.
2. Inner join com o Censo daquele ano pela chave `CO_ENTIDADE`.
3. Adição da coluna `ANO`.

**Nota:** Para o ano de **2023**, não existe `VL_PROJECAO_2023` nos dados do INEP (as metas projetadas vão apenas até 2021). Nesses casos, a coluna `IDEB_META` é preenchida com `NaN`.

In [8]:
lista_dfs_anuais = []

for ano in ANOS:
    # Carrega o Censo filtrado por PB daquele ano
    caminho_censo = os.path.join(DADOS_TRATADOS, f'censo_pb_{ano}.csv')
    df_censo_ano = pd.read_csv(caminho_censo, low_memory=False)
    qtd_antes = len(df_censo_ano)
    
    # Realiza o merge
    df_merged = fazer_merge_ano(df_censo_ano, df_ideb_limpo, ano)
    qtd_depois = len(df_merged)
    qtd_ideb_nan = df_merged['IDEB'].isna().sum()
    
    print(f'Ano {ano}: Censo={qtd_antes:,} -> Merge={qtd_depois:,} | IDEB NaN={qtd_ideb_nan:,}')
    
    lista_dfs_anuais.append(df_merged)

Ano 2015: Censo=7,135 -> Merge=1,823 | IDEB NaN=906
Ano 2017: Censo=6,901 -> Merge=1,829 | IDEB NaN=859
Ano 2019: Censo=6,508 -> Merge=1,814 | IDEB NaN=731
Ano 2021: Censo=5,638 -> Merge=1,798 | IDEB NaN=1,043
Ano 2023: Censo=5,479 -> Merge=1,790 | IDEB NaN=748


---
## 6. Montagem do Painel Final

A função `montar_painel()` empilha os DataFrames dos 5 anos em um único dataset, onde cada linha representa uma escola em um determinado ano. Em seguida:
- Remove linhas onde a nota do `IDEB` é `NaN` (escola existia no Censo e no cadastro do IDEB, mas não teve nota calculada naquele ciclo).
- Cria a variável alvo `ATINGIU_META`:
  - `1.0` se `IDEB >= IDEB_META`
  - `0.0` se `IDEB < IDEB_META`
  - `NaN` se `IDEB_META` não estiver disponível (caso de 2023 e escolas sem projeção)

In [9]:
df_painel = montar_painel(lista_dfs_anuais)

print('=== RESUMO FINAL DO DATASET ===')
print(f'Shape final: {df_painel.shape}')
print(f'Escolas únicas (CO_ENTIDADE): {df_painel["CO_ENTIDADE"].nunique()}')
print()
print('Linhas por ano:')
print(df_painel['ANO'].value_counts().sort_index())
print()
print('Coluna ATINGIU_META:')
qtd_preenchidos = df_painel['ATINGIU_META'].notna().sum()
qtd_nan = df_painel['ATINGIU_META'].isna().sum()
print(f'  Preenchidos (0 ou 1): {qtd_preenchidos}')
print(f'  Faltantes (NaN): {qtd_nan}')
print()
print('Distribuição:')
print(df_painel['ATINGIU_META'].value_counts(dropna=False))

=== RESUMO FINAL DO DATASET ===
Shape final: (4767, 457)
Escolas únicas (CO_ENTIDADE): 1597

Linhas por ano:
ANO
2015     917
2017     970
2019    1083
2021     755
2023    1042
Name: count, dtype: int64

Coluna ATINGIU_META:
  Preenchidos (0 ou 1): 3343
  Faltantes (NaN): 1424

Distribuição:
ATINGIU_META
1.0    1861
0.0    1482
NaN    1424
Name: count, dtype: int64


---
## 7. Exportação do Dataset Final

O dataset em formato de painel é salvo em `dados_tratados/dataset_final.csv`, contendo todas as colunas de infraestrutura do Censo, as notas do IDEB, a meta projetada e a variável alvo `ATINGIU_META`. A seleção de features específicas será realizada na etapa seguinte (notebook 02).

In [10]:
output_path = os.path.join(DADOS_TRATADOS, 'dataset_final.csv')
df_painel.to_csv(output_path, index=False, encoding='utf-8')
print(f'Dataset final salvo em: {output_path}')

Dataset final salvo em: /Users/anne/Documents/UFPB/ideb-previsao/dados_tratados/dataset_final.csv
